# A RAG-based Research Assistant

A RAG-based knowledge worker that enables users to chat with research documents.

The assistant uses the Attention Is All You Need paper as the default knowledge base and retrieves relevant context to answer user questions.

Key features;
- Document ingestion for multiple formats (PDF, Markdown, TXT, HTML)

- Semantic search using sentence-transformers/all-MiniLM-L6-v2 embeddings

- Vector storage with Chroma

- Cross-encoder reranking for improved retrieval accuracy

- Conversation-aware query rewriting

- Streaming responses for better user experience

- Context panel displaying retrieved sources

It is designed to be modular and easily extensible for additional documents or knowledge bases.

This notebook combines:
- **Ingestion**: Loading documents, creating chunks, and building embeddings
- **Answer**: RAG-based question answering with context retrieval
- **App**: Gradio interface for interactive chat

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredMarkdownLoader,
    UnstructuredHTMLLoader
)

from sentence_transformers import CrossEncoder

In [ ]:
MODEL_NAME="openai/gpt-oss-120b:free"
EMBED_MODEL = "text-embedding-3-large"

KNOWLEDGE_BASE = "nj-knowledge-base"
VECTOR_DB_DIR = "nj-vectorstore"

DEFAULT_PAPER = "attention_is_all_you_need.pdf"

load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found")


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32}
)

llm = ChatOpenAI(
    model=MODEL_NAME,
    streaming=True,
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

Path(KNOWLEDGE_BASE).mkdir(exist_ok=True)
Path(VECTOR_DB_DIR).mkdir(exist_ok=True)

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

SYSTEM_PROMPT = """

    You are an AI research assistant.

    You answer questions using ONLY the provided context from the research paper 
    "Attention Is All You Need".

    Rules:
    - Use only the provided context
    - If the answer is not in the context say:
    "I could not find this information in the paper."
    - Do NOT invent information
    - Explain concepts clearly

    CONTEXT
    -------
    {context}

    QUESTION
    --------
    {question}
    """

In [ ]:
# Document Loaders  (Multiple File Types)

def load_file(file_path):

    ext = Path(file_path).suffix.lower()

    if ext == ".pdf":
        loader = PyPDFLoader(file_path)

    elif ext == ".md":
        loader = UnstructuredMarkdownLoader(file_path)

    elif ext == ".txt":
        loader = TextLoader(file_path)

    elif ext == ".html":
        loader = UnstructuredHTMLLoader(file_path)

    else:
        return []

    docs = loader.load()

    for d in docs:
        d.metadata["source"] = str(file_path)

    return docs

In [ ]:
def fetch_documents():

    docs = []

    for file in Path(KNOWLEDGE_BASE).glob("*"):
        docs.extend(load_file(file))

    print(f"Loaded {len(docs)} documents")

    return docs



def create_chunks(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=120
    )

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    print(f"Created {len(chunks)} chunks")

    return chunks



In [ ]:
# create vector store

def create_vectorstore(chunks):

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=VECTOR_DB_DIR
    )

    vectorstore.persist()

    return vectorstore



def load_vectorstore():

    vectorstore = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=embeddings
    )

    return vectorstore



In [ ]:
if not Path(VECTOR_DB_DIR).exists() or len(list(Path(VECTOR_DB_DIR).glob("*"))) == 0:

    print("Creating vector store...")

    docs = fetch_documents()
    chunks = create_chunks(docs)

    vectorstore = create_vectorstore(chunks)

else:

    print("Loading existing vector store...")

    vectorstore = load_vectorstore()

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

In [ ]:
def rerank_documents(query, docs, top_k=4):

    pairs = [[query, doc.page_content] for doc in docs]

    scores = reranker.predict(pairs)

    scored = list(zip(docs, scores))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in scored[:top_k]]


# Conversation-Aware Query

def build_query(question, history):

    history_text = ""

    for turn in history[-4:]:
        history_text += f"{turn['role']}: {turn['content']}\n"

    query_prompt = f"""
        Rewrite the following user question into a standalone question 
        for document retrieval.

        Conversation:
        {history_text}

        User question:
        {question}
        """

    response = llm.invoke(query_prompt)

    return response.content

In [ ]:
def fetch_context(question, history):

    search_query = build_query(question, history)

    docs = retriever.invoke(search_query)

    docs = rerank_documents(question, docs)

    return docs




def combine_context(docs):

    blocks = []

    for i, doc in enumerate(docs):

        source = doc.metadata.get("source", "unknown")

        block = f"""
            SOURCE {i+1}
            File: {source}

            {doc.page_content}
            """
        blocks.append(block)

    return "\n".join(blocks)

In [ ]:
# streaming response generation

def answer_question(question, history):

    docs = fetch_context(question, history)

    context = combine_context(docs)

    prompt = SYSTEM_PROMPT.format(
        context=context,
        question=question
    )

    response = ""

    for chunk in llm.stream(prompt):

        if chunk.content:
            response += chunk.content
            yield response, docs
            
            
# Display Retrieved Context   
   
def format_context(docs):

    text = "### Retrieved Context\n\n"

    for d in docs:
        src = d.metadata.get("source", "unknown")

        text += f"**Source:** {src}\n\n"
        text += d.page_content[:500] + "\n\n---\n\n"

    return text

## Ingestion Step (Run Once)

Run this cell before launching the app.

In [ ]:
docs = fetch_documents()

chunks = create_chunks(docs)

vectorstore = create_vectorstore(chunks)

print("Ingestion complete")

In [ ]:
def chat(message, history):

    history = history or []

    for response, docs in answer_question(message, history):

        yield response, format_context(docs)
        
        

with gr.Blocks() as demo:

    gr.Markdown("# 📚 Attention Is All You Need – RAG Assistant")

    with gr.Row():

        chatbot = gr.ChatInterface(
            fn=chat,
            type="messages"
        )

        context_panel = gr.Markdown(
            "Retrieved context will appear here"
        )

demo.launch()